In [1]:
import sys
sys.path.append('../src')

In [2]:
from gso.gso_learning import cgl_fit, glasso_fit

In [5]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx
import wntr
from scipy import sparse
from scipy.sparse.linalg import eigsh
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)

/home/en4624/ghorg/graphdatahub/NetworkInference/.venv/lib/python3.12/site-packages/wntr/epanet/toolkit.py:13: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_filename


In [7]:
def simulate_network3_hydraulics():
    """
    Simulate EPANET Network 3 hydraulics over 168 hours with varying demands
    Returns time series data for graph signal processing
    """
    # Load EPANET Network 3
    inp_file = 'Net3.inp'
    wn = wntr.network.WaterNetworkModel(inp_file)
    
    # Set simulation duration to 168 hours (7 days)
    wn.options.time.duration = 168 * 3600  # Convert to seconds
    wn.options.time.hydraulic_timestep = 3600  # 1 hour timestep
    wn.options.time.report_timestep = 3600  # Report every hour
    
    print(f"Network loaded: {wn.name}")
    print(f"Number of junctions: {wn.num_junctions}")
    print(f"Number of pipes: {wn.num_pipes}")
    print(f"Simulation duration: {wn.options.time.duration/3600} hours")
    
    # Create demand multiplier patterns to simulate varying demands
    # This creates realistic demand variations over the week
    time_steps = int(wn.options.time.duration / wn.options.time.hydraulic_timestep) + 1
    
    # Create weekly demand pattern with daily cycles
    hours = np.arange(time_steps)
    daily_pattern = 0.8 + 0.4 * np.sin(2 * np.pi * hours / 24)  # Daily cycle
    weekly_pattern = 1.0 + 0.2 * np.sin(2 * np.pi * hours / (24 * 7))  # Weekly cycle
    demand_multipliers = daily_pattern * weekly_pattern
    
    # Add some random variation to make it more realistic
    demand_multipliers += 0.1 * np.random.randn(len(demand_multipliers))
    demand_multipliers = np.clip(demand_multipliers, 0.3, 2.0)  # Keep reasonable bounds
    
    # Apply demand multipliers to all junctions
    for i, junction_name in enumerate(wn.junction_name_list):
        junction = wn.get_node(junction_name)
        if junction.demand_timeseries_list:
            # Modify existing demand pattern
            base_demand = junction.demand_timeseries_list[0].base_value
            new_demands = base_demand * demand_multipliers
            
            # Create new demand timeseries
            times = hours * 3600  # Convert to seconds
            demand_pattern = list(zip(times, new_demands))
            junction.demand_timeseries_list[0].base_value = 1.0
            wn.add_pattern(f'pattern_{i}', demand_multipliers.tolist())
            junction.demand_timeseries_list[0].pattern_name = f'pattern_{i}'
    
    # Run hydraulic simulation using WNTR simulator for better control
    sim = wntr.sim.WNTRSimulator(wn)
    results = sim.run_sim()
    
    print("Simulation completed successfully!")
    
    # Extract node pressure signals (these will be our graph signals)
    pressure_data = results.node['pressure']
    
    # Remove tanks and reservoirs, keep only junctions for graph signal analysis
    junction_names = wn.junction_name_list
    pressure_signals = pressure_data[junction_names]
    
    print(f"Extracted pressure signals for {len(junction_names)} junctions")
    print(f"Time series length: {len(pressure_signals)} time steps")
    
    return wn, pressure_signals, junction_names

def create_network_laplacian(wn, weight_type='uniform'):
    """
    Create network Laplacian matrix from water network
    
    Parameters:
    - wn: Water network model
    - weight_type: 'uniform', 'length', 'diameter', or 'resistance'
    """
    # Get NetworkX graph representation
    G = wn.get_graph()
    
    # Convert to undirected graph for Laplacian computation
    G_undirected = G.to_undirected()
    
    # Create junction mapping (exclude tanks and reservoirs)
    junction_names = wn.junction_name_list
    node_mapping = {name: i for i, name in enumerate(junction_names)}
    
    # Create subgraph with only junctions
    junction_graph = G_undirected.subgraph(junction_names).copy()
    
    # Relabel nodes to use indices
    junction_graph = nx.relabel_nodes(junction_graph, node_mapping)
    
    # Compute weights based on pipe properties
    if weight_type == 'uniform':
        # Uniform weights (simple connectivity)
        L = nx.laplacian_matrix(junction_graph, weight=None).astype(float)
    elif weight_type == 'length':
        # Weight by inverse pipe length
        for u, v, data in junction_graph.edges(data=True):
            length = wn.get_link(data.get('name', '')).length
            if length > 0:
                data['weight'] = 1.0 / length
            else:
                data['weight'] = 1.0
        L = nx.laplacian_matrix(junction_graph, weight='weight').astype(float)
    elif weight_type == 'diameter':
        # Weight by pipe diameter
        for u, v, data in junction_graph.edges(data=True):
            diameter = wn.get_link(data.get('name', '')).diameter
            if diameter > 0:
                data['weight'] = diameter
            else:
                data['weight'] = 1.0
        L = nx.laplacian_matrix(junction_graph, weight='weight').astype(float)
    elif weight_type == 'resistance':
        # Weight by inverse hydraulic resistance
        for u, v, data in junction_graph.edges(data=True):
            link = wn.get_link(data.get('name', ''))
            # Simplified resistance calculation
            if link.length > 0 and link.diameter > 0:
                resistance = link.length / (link.diameter ** 4)
                data['weight'] = 1.0 / resistance if resistance > 0 else 1.0
            else:
                data['weight'] = 1.0
        L = nx.laplacian_matrix(junction_graph, weight='weight').astype(float)
    
    return L.toarray()

def compute_gft_power_spectrum(signals, laplacian):
    """
    Compute Graph Fourier Transform power spectrum for given signals and Laplacian
    
    Parameters:
    - signals: pandas DataFrame with time series signals (time x nodes)
    - laplacian: Laplacian matrix (nodes x nodes)
    
    Returns:
    - eigenvalues: Graph frequencies (sorted)
    - power_spectrum: Average power spectrum across all time steps
    - gft_coefficients: GFT coefficients for all signals
    """
    print("Computing eigendecomposition of Laplacian...")
    
    # Compute eigendecomposition of Laplacian
    eigenvalues, eigenvectors = np.linalg.eigh(laplacian)
    
    # Sort by eigenvalues (graph frequencies)
    sort_idx = np.argsort(eigenvalues)
    eigenvalues = eigenvalues[sort_idx]
    eigenvectors = eigenvectors[:, sort_idx]
    
    print(f"Computed {len(eigenvalues)} eigenvalues/eigenvectors")
    
    # Convert signals to numpy array
    signal_matrix = signals.values  # Shape: (time_steps, num_nodes)
    
    print("Computing GFT coefficients...")
    
    # Compute GFT coefficients for each time step
    # GFT coefficients = eigenvectors^T * signal
    gft_coefficients = np.dot(signal_matrix, eigenvectors)  # Shape: (time_steps, num_freqs)
    
    # Compute power spectrum (squared magnitude of coefficients)
    power_spectrum = np.abs(gft_coefficients) ** 2
    
    # Average power across all time steps
    avg_power_spectrum = np.mean(power_spectrum, axis=0)
    
    print("GFT power spectrum computation completed!")
    
    return eigenvalues, avg_power_spectrum, gft_coefficients

def analyze_signal_properties(eigenvalues, power_spectrum, gft_coefficients):
    """
    Analyze compressibility and smoothness of signals
    """
    print("\n" + "="*50)
    print("SIGNAL ANALYSIS RESULTS")
    print("="*50)
    
    # 1. Energy concentration analysis
    total_energy = np.sum(power_spectrum)
    cumulative_energy = np.cumsum(power_spectrum) / total_energy
    
    # Find how many coefficients contain 90% and 95% of energy
    energy_90_idx = np.where(cumulative_energy >= 0.90)[0][0] + 1
    energy_95_idx = np.where(cumulative_energy >= 0.95)[0][0] + 1
    
    print(f"Energy concentration:")
    print(f"  - 90% of energy in first {energy_90_idx}/{len(eigenvalues)} coefficients ({energy_90_idx/len(eigenvalues)*100:.1f}%)")
    print(f"  - 95% of energy in first {energy_95_idx}/{len(eigenvalues)} coefficients ({energy_95_idx/len(eigenvalues)*100:.1f}%)")
    
    # 2. Low-frequency dominance
    low_freq_threshold = 0.1 * np.max(eigenvalues)
    low_freq_mask = eigenvalues <= low_freq_threshold
    low_freq_energy = np.sum(power_spectrum[low_freq_mask])
    low_freq_ratio = low_freq_energy / total_energy
    
    print(f"\nLow-frequency analysis:")
    print(f"  - Low-frequency energy ratio: {low_freq_ratio:.3f}")
    print(f"  - Number of low-frequency components: {np.sum(low_freq_mask)}/{len(eigenvalues)}")
    
    # 3. Compressibility metrics
    # Compute effective rank (number of significant coefficients)
    normalized_spectrum = power_spectrum / total_energy
    entropy = -np.sum(normalized_spectrum * np.log(normalized_spectrum + 1e-12))
    max_entropy = np.log(len(eigenvalues))
    effective_rank = np.exp(entropy)
    
    print(f"\nCompressibility metrics:")
    print(f"  - Spectral entropy: {entropy:.3f} (max: {max_entropy:.3f})")
    print(f"  - Effective rank: {effective_rank:.1f}/{len(eigenvalues)}")
    print(f"  - Compressibility ratio: {effective_rank/len(eigenvalues):.3f}")
    
    # 4. Smoothness assessment
    # Signals are smooth if most energy is in low eigenvalues
    if low_freq_ratio > 0.7:
        smoothness = "HIGH - Signals are smooth on the graph"
    elif low_freq_ratio > 0.4:
        smoothness = "MODERATE - Signals have some smoothness"
    else:
        smoothness = "LOW - Signals are not smooth on the graph"
    
    print(f"\nSmoothness assessment: {smoothness}")
    
    # 5. Compressibility assessment
    if energy_90_idx / len(eigenvalues) < 0.2:
        compressibility = "HIGH - Signals are highly compressible"
    elif energy_90_idx / len(eigenvalues) < 0.5:
        compressibility = "MODERATE - Signals show some compressibility"
    else:
        compressibility = "LOW - Signals are not well compressible"
    
    print(f"Compressibility assessment: {compressibility}")
    
    return {
        'energy_90_idx': energy_90_idx,
        'energy_95_idx': energy_95_idx,
        'low_freq_ratio': low_freq_ratio,
        'entropy': entropy,
        'effective_rank': effective_rank,
        'compressibility_ratio': effective_rank/len(eigenvalues)
    }

def plot_results(eigenvalues, power_spectrum, gft_coefficients, analysis_results):
    """
    Create comprehensive visualization of results
    """
    fig, axes = plt.subplots(2, 2, figsize=(15, 12))
    
    # 1. Power spectrum
    axes[0, 0].semilogy(eigenvalues, power_spectrum, 'b-', linewidth=2)
    axes[0, 0].set_xlabel('Graph Frequency (Eigenvalue)')
    axes[0, 0].set_ylabel('Average Power')
    axes[0, 0].set_title('GFT Power Spectrum')
    axes[0, 0].grid(True, alpha=0.3)
    
    # Mark 90% energy threshold
    energy_90_idx = analysis_results['energy_90_idx']
    if energy_90_idx < len(eigenvalues):
        axes[0, 0].axvline(eigenvalues[energy_90_idx], color='r', linestyle='--', 
                          label=f'90% Energy ({energy_90_idx} coeffs)')
        axes[0, 0].legend()
    
    # 2. Cumulative energy
    cumulative_energy = np.cumsum(power_spectrum) / np.sum(power_spectrum)
    axes[0, 1].plot(range(len(eigenvalues)), cumulative_energy, 'g-', linewidth=2)
    axes[0, 1].axhline(0.9, color='r', linestyle='--', label='90% Energy')
    axes[0, 1].axhline(0.95, color='orange', linestyle='--', label='95% Energy')
    axes[0, 1].set_xlabel('Fourier Coefficient Index')
    axes[0, 1].set_ylabel('Cumulative Energy Fraction')
    axes[0, 1].set_title('Energy Concentration')
    axes[0, 1].grid(True, alpha=0.3)
    axes[0, 1].legend()
    
    # 3. Sample time series of GFT coefficients
    time_steps = min(100, gft_coefficients.shape[0])  # Show first 100 time steps
    for i in range(min(5, gft_coefficients.shape[1])):  # Show first 5 frequencies
        axes[1, 0].plot(gft_coefficients[:time_steps, i], label=f'Freq {i}')
    axes[1, 0].set_xlabel('Time Step')
    axes[1, 0].set_ylabel('GFT Coefficient')
    axes[1, 0].set_title('GFT Coefficients Over Time (First 5 Frequencies)')
    axes[1, 0].legend()
    axes[1, 0].grid(True, alpha=0.3)
    
    # 4. Eigenvalue distribution
    axes[1, 1].hist(eigenvalues, bins=30, alpha=0.7, color='purple', edgecolor='black')
    axes[1, 1].set_xlabel('Eigenvalue')
    axes[1, 1].set_ylabel('Frequency')
    axes[1, 1].set_title('Distribution of Graph Frequencies')
    axes[1, 1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

def analyze_custom_laplacian(pressure_signals, custom_laplacian):
    """
    Analyze signals with a custom Laplacian matrix
    
    Parameters:
    - pressure_signals: pandas DataFrame with pressure time series
    - custom_laplacian: Custom Laplacian matrix (numpy array)
    
    Returns:
    - Dictionary with analysis results
    """
    print("Analyzing signals with custom Laplacian...")
    
    # Compute GFT power spectrum
    eigenvalues, power_spectrum, gft_coefficients = compute_gft_power_spectrum(
        pressure_signals, custom_laplacian)
    
    # Analyze properties
    analysis_results = analyze_signal_properties(eigenvalues, power_spectrum, gft_coefficients)
    
    # Create plots
    plot_results(eigenvalues, power_spectrum, gft_coefficients, analysis_results)
    
    return {
        'eigenvalues': eigenvalues,
        'power_spectrum': power_spectrum,
        'gft_coefficients': gft_coefficients,
        'analysis': analysis_results
    }

def create_binary_adjacency_matrix(wn):
    """
    Generate binary adjacency matrix (0/1) for junctions only
    Returns numpy array of shape (97, 97)
    """
    # Get directed multigraph from WNTR
    G = wn.get_graph()
    
    # Extract junction subgraph and convert to undirected simple graph
    junction_names = wn.junction_name_list
    junction_subgraph = G.subgraph(junction_names).to_undirected()
    simple_graph = nx.Graph(junction_subgraph)  # Merge multiple edges
    
    # Create binary adjacency matrix
    A = nx.adjacency_matrix(simple_graph, weight=None)
    
    return A.toarray()

In [8]:
wn, pressure_signals, junction_names = simulate_network3_hydraulics()

print(f"Signal shape: {pressure_signals.shape}")
print(f"First 5 junctions: {pressure_signals.columns[:5].tolist()}")
print(f"Time range: {pressure_signals.index[0]} to {pressure_signals.index[-1]}")

# Verify matrix properties
A = create_binary_adjacency_matrix(wn)
assert A.shape == (97, 97)
assert np.allclose(A, A.T)
assert np.sum(np.diag(A)) == 0

Network loaded: Net3.inp
Number of junctions: 92
Number of pipes: 117
Simulation duration: 168.0 hours


TypeError: Array of type 'long' required.  A 'unknown type' was given

In [ ]:
sample_cov = np.cov(pressure_signals, rowvar=True, bias=False, ddof=1)
L = cgl_fit(sample_cov, A_mask=A)

eigenvalues, power_spectrum, gft_coefficients = compute_gft_power_spectrum(pressure_signals, L)

analysis_results = analyze_signal_properties(eigenvalues, power_spectrum, gft_coefficients)
            
plot_results(eigenvalues, power_spectrum, gft_coefficients, analysis_results)